# AegisForge-AI — Router Flow Visualization

This notebook recreates the routing flow represented by the uploaded AegisForge router code: deterministic fast paths, LLM fallback routing, supervisor processing, human approval, and chief review.

The visualization is designed to work directly inside Jupyter/Colab.

In [ ]:
# Install only if required in your environment
# %pip install -q langgraph langchain-core pygraphviz

from typing import TypedDict, Literal
from IPython.display import display, Markdown, Image

try:
    from langgraph.graph import StateGraph, START, END
except ImportError:
    raise ImportError("Install LangGraph first: %pip install -q langgraph")

In [ ]:
class AgentState(TypedDict, total=False):
    question: str
    route: str
    retry_count: int
    safety_flag: bool
    human_approved: bool
    human_feedback: str
    result: str


# -----------------------------
# Router logic from the source
# -----------------------------
_PURE_GREETINGS = {
    "hi", "hello", "hey", "thanks", "thank you", "bye", "goodbye",
    "good morning", "good afternoon", "good evening", "good night",
    "namaste", "howdy",
}

_TECHNICAL_INDICATORS = {
    "asme", "ug-27", "ug-28", "api", "api 579", "gfr", "cvc", "rbi",
    "mawp", "design pressure", "wall thickness", "corrosion allowance",
    "corrosion rate", "allowable stress", "joint efficiency", "inside radius",
    "pressure vessel", "separator drum", "heat exchanger", "piping", "flange",
    "nozzle", "shell", "head", "remaining life", "calculate", "compliance",
    "sandbox", "python script", "docker", "inspection", "ocr", "extract",
    "simulate", "mpa", "kpa", "psi", "bar", "mm/yr", "n/mm2",
}

def route_question(state: AgentState) -> str:
    question = state["question"].strip()
    q_lower = question.lower()
    words = q_lower.split()

    # 1. Technical indicator -> supervisor
    if any(ind in q_lower for ind in _TECHNICAL_INDICATORS):
        return "supervisor"

    # 2. Engineering units / numbers -> supervisor
    import re
    if re.search(r'\\d+\\s*(?:mpa|kpa|bar|psi|mm|cm|m|°c|c|yr|year|kg|kn|%)', q_lower):
        return "supervisor"

    # 3. Long prompt -> supervisor
    if len(words) > 12:
        return "supervisor"

    # 4. Pure greeting -> direct answer
    if q_lower in _PURE_GREETINGS:
        return "direct_answer"

    # 5. Short greeting phrase -> direct answer
    if len(words) <= 4 and any(q_lower.startswith(g) for g in _PURE_GREETINGS):
        return "direct_answer"

    # 6. Short identity/capability question -> direct answer
    if len(words) <= 8 and any(
        p in q_lower for p in ["who are you", "what can you do", "help me with what", "how are you"]
    ):
        return "direct_answer"

    # 7. In the real system this is the router LLM fallback.
    # For visualization/demo purposes we default to supervisor,
    # matching the source code's failure-safe behavior.
    return "supervisor"


def router_node(state: AgentState):
    route = route_question(state)
    return {"route": route}


def direct_answer_node(state: AgentState):
    return {"result": "Direct conversational answer"}


def supervisor_node(state: AgentState):
    return {"result": "Supervisor processes engineering/technical task"}


def retry_or_review(state: AgentState) -> str:
    # Represents the safety/retry decision that can lead to HITL.
    if state.get("safety_flag", False) or state.get("retry_count", 0) >= 3:
        return "human_gate"
    return "chief_reviewer"


def human_approval_gate(state: AgentState):
    # Visualization-only representation of interrupt(...).
    # In production, replace this node with the actual LangGraph interrupt gate.
    return {
        "human_approved": True,
        "human_feedback": "Engineer sign-off",
        "retry_count": 0,
    }


def after_human_gate(state: AgentState) -> str:
    return "chief_reviewer" if state.get("human_approved") else "supervisor"


def chief_reviewer_node(state: AgentState):
    return {"result": "Chief reviewer validates the final result"}

In [ ]:
# Build the graph
builder = StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("direct_answer", direct_answer_node)
builder.add_node("supervisor", supervisor_node)
builder.add_node("human_gate", human_approval_gate)
builder.add_node("chief_reviewer", chief_reviewer_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "direct_answer": "direct_answer",
        "supervisor": "supervisor",
    },
)

builder.add_edge("direct_answer", END)

# Supervisor can finish normally or trigger human review.
builder.add_conditional_edges(
    "supervisor",
    retry_or_review,
    {
        "human_gate": "human_gate",
        "chief_reviewer": "chief_reviewer",
    },
)

# Human approval routes to chief reviewer when approved,
# otherwise back to supervisor.
builder.add_conditional_edges(
    "human_gate",
    after_human_gate,
    {
        "chief_reviewer": "chief_reviewer",
        "supervisor": "supervisor",
    },
)

builder.add_edge("chief_reviewer", END)

graph = builder.compile()
print("Graph compiled successfully.")

## Flow diagram

The following is the same logical structure represented by the notebook graph.

In [ ]:
# Render LangGraph's Mermaid representation directly in the notebook.
mermaid = graph.get_graph().draw_mermaid()
display(Markdown("```mermaid\n" + mermaid + "\n```"))

## Human-readable flow

```text
START
  │
  ▼
┌──────────────┐
│    ROUTER    │
└──────┬───────┘
       │
       ├── direct_answer ─────────────► DIRECT ANSWER ──► END
       │
       └── supervisor
               │
               ▼
         ┌─────────────┐
         │  SUPERVISOR │
         └──────┬──────┘
                │
        ┌───────┴────────┐
        │                │
        ▼                ▼
   HUMAN GATE       CHIEF REVIEWER
        │                │
   ┌────┴────┐           │
   │         │           │
   ▼         ▼           ▼
 APPROVED  REJECTED    END
   │         │
   ▼         └──────────► SUPERVISOR
 CHIEF REVIEWER
   │
   ▼
  END
```

In [ ]:
# Optional: visualize the router behavior with sample inputs.
examples = [
    "hello",
    "Namaste",
    "Who are you?",
    "Calculate wall thickness for a pressure vessel",
    "Check ASME Section VIII compliance",
    "What is the weather today?",
    "Evaluate remaining life at 12 MPa",
]

for q in examples:
    route = route_question({"question": q})
    print(f"{q:<55} -> {route}")

## Important implementation note

The uploaded source explicitly uses an LLM fallback for ambiguous short queries and a real `interrupt(...)` human-approval gate. This notebook keeps those two points as visualization/demo behavior; replace the corresponding functions with your actual `router_llm`, `interrupt`, supervisor, and reviewer implementations when integrating it into the project. fileciteturn0file0L117-L132